**Inisialisasi SparkSession**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, desc, avg, sum, count

spark = SparkSession.builder \
    .appName("Tugas4_BigData") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession berhasil dibuat!")

26/09/16 18:02:40 WARN Utils: Your hostname, haris resolves to a loopback address: 127.0.1.1; using 192.168.1.55 instead (on interface wlp1s0)
26/09/16 18:02:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 18:02:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/16 18:02:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat!


**A. Membaca dan Eksplorasi Awal (15%)**

In [2]:
path_hdfs = "hdfs://localhost:9000/user/haris/tugas4/transaksi_september_2026.csv"
df = spark.read.csv(path_hdfs, header=True, inferSchema=True)

# 1. Tampilkan Schema
print("--- Schema Data ---")
df.printSchema()

# 2. Tampilkan Jumlah Baris
print(f"Total baris data: {df.count()}")

# 3. Tampilkan 10 Baris Pertama
print("\n--- 10 Baris Pertama ---")
df.show(10)

--- Schema Data ---
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Total baris data: 1000

--- 10 Baris Pertama ---
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26

**B. Menangani Data Kosong (15%)**

In [3]:
# Tampilkan jumlah rating kosong
null_rating = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {null_rating}")

# Menangani data kosong dengan df.na.drop()
df_clean = df.na.drop(subset=["rating"])

print(f"Jumlah baris setelah data kosong dibuang: {df_clean.count()}")

Jumlah baris dengan rating kosong: 204
Jumlah baris setelah data kosong dibuang: 796


**C. Transformasi Data (20%)**

In [4]:
# Menambahkan kolom total_pendapatan dan tier_transaksi
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

df_transformed.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

**D. Analisis dengan GroupBy (30%)**

In [6]:
# 1. Kategori dengan total_pendapatan tertinggi
print("--- 1. Kategori dengan Total Pendapatan Tertinggi ---")
df_transformed.groupBy("kategori") \
              .agg(sum("total_pendapatan").alias("total_pendapatan_kategori")) \
              .orderBy(desc("total_pendapatan_kategori")) \
              .show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("--- 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ---")
df_transformed.filter(col("tier_transaksi") == "Besar") \
              .groupBy("kota") \
              .count() \
              .orderBy(desc("count")) \
              .show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("--- 3. Rata-rata Rating per Metode Pembayaran ---")
df_transformed.groupBy("metode_pembayaran") \
              .agg(avg("rating").alias("rata_rata_rating")) \
              .show()

--- 1. Kategori dengan Total Pendapatan Tertinggi ---
+------------+-------------------------+
|    kategori|total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|                108285000|
+------------+-------------------------+
only showing top 1 row

--- 2. Kota dengan Transaksi Tier 'Besar' Terbanyak ---
+----+-----+
|kota|count|
+----+-----+
|Solo|   74|
+----+-----+
only showing top 1 row

--- 3. Rata-rata Rating per Metode Pembayaran ---
+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|     Kartu Kredit|4.109947643979058|
|         E-Wallet|4.135678391959799|
+-----------------+-----------------+



**E. Menyimpan Hasil ke HDFS (20%)**

In [7]:
output_hdfs_path = "hdfs://localhost:9000/user/haris/tugas4/hasil_transaksi_september"

# Menyimpan DataFrame hasil transformasi ke HDFS
df_transformed.write \
              .mode("overwrite") \
              .option("header", "true") \
              .csv(output_hdfs_path)

print("Berhasil menyimpan data ke HDFS!")

Berhasil menyimpan data ke HDFS!


In [9]:
##Verifikasi
!hdfs dfs -ls /user/haris/tugas4/hasil_transaksi_september

Found 2 items
-rw-r--r--   3 haris supergroup          0 2026-09-16 18:18 /user/haris/tugas4/hasil_transaksi_september/_SUCCESS
-rw-r--r--   3 haris supergroup      77337 2026-09-16 18:18 /user/haris/tugas4/hasil_transaksi_september/part-00000-3fff6864-0ae5-449c-893e-072f137ba759-c000.csv


In [10]:
!hdfs dfs -cat /user/haris/tugas4/hasil_transaksi_september/part-*.csv | head -n 10

order_id,tanggal,kategori,kota,unit_terjual,harga_satuan,metode_pembayaran,rating,total_pendapatan,tier_transaksi
ORD-3000,2026-09-02T00:00:00.000+07:00,Rumah Tangga,Yogyakarta,3,90000,COD,4.0,270000,Kecil
ORD-3001,2026-09-04T00:00:00.000+07:00,Makanan & Minuman,Solo,3,200000,E-Wallet,5.0,600000,Besar
ORD-3002,2026-09-26T00:00:00.000+07:00,Kesehatan & Kecantikan,Semarang,8,60000,E-Wallet,3.0,480000,Kecil
ORD-3003,2026-09-09T00:00:00.000+07:00,Makanan & Minuman,Semarang,6,350000,Transfer Bank,4.0,2100000,Besar
ORD-3004,2026-09-10T00:00:00.000+07:00,Rumah Tangga,Yogyakarta,10,60000,E-Wallet,4.0,600000,Besar
ORD-3005,2026-09-09T00:00:00.000+07:00,Fashion,Purworejo,5,20000,E-Wallet,4.0,100000,Kecil
ORD-3006,2026-09-19T00:00:00.000+07:00,Makanan & Minuman,Yogyakarta,2,20000,COD,5.0,40000,Kecil
ORD-3008,2026-09-06T00:00:00.000+07:00,Fashion,Semarang,7,20000,Kartu Kredit,5.0,140000,Kecil
ORD-3009,2026-09-21T00:00:00.000+07:00,Elektronik,Yogyakarta,10,90000,Transfer Bank,3.0,900000,Besar
cat: 